In [18]:
!pip install langchain -U langchain-community faiss-cpu langchain-openai tiktoken unstructured selenium newspaper3k textstat
!pip install sentence-transformers==2.2.2
!pip install InstructorEmbedding
!pip install pypdf transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 kB 903.2 kB/s eta 0:00:00
  Attempting uninstall: textstat
    Found existing installation: textstat 0.7.3
    Uninstalling textstat-0.7.3:
      Successfully uninstalled textstat-0.7.3


In [19]:
import os
from google.colab import drive, userdata
import pickle
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
import openai
from transformers import pipeline

# Mount Google Drive
drive.mount('/content/gdrive', force_remount=True)
root_dir = "/content/gdrive/MyDrive/WAI_project/"

# Set environment variables for OpenAI
os.environ["OPENAI_API_KEY"] = "OPEN AI KEY"


# Set Hugging Face token
hf_token = userdata.get('HF_TOKEN')
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

Mounted at /content/gdrive


In [24]:
## Step 1: Ingestion and Chunking
# Initialize text_splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)
def ingest_and_chunk_pdfs():
    """
    Ingests PDF documents and chunks the content into smaller segments.

    Returns:
    - list: List of text chunks.
    """

    loader = DirectoryLoader(f"{root_dir}", glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()

    # Debug: Print the number of documents loaded
    print(f"Number of documents loaded: {len(documents)}")
    for i, doc in enumerate(documents[:3]):  # Print first 3 documents for verification
        print(f"Document {i+1}:")
        print(doc.page_content[:500])  # Print first 500 characters of each document

    texts = text_splitter.split_documents(documents)

    # Debug: Print the number of text chunks created
    print(f"Number of text chunks: {len(texts)}")
    for i, text in enumerate(texts[:3]):  # Print first 3 chunks for verification
        print(f"Chunk {i+1}:")
        print(text.page_content[:500])  # Print first 500 characters of each chunk
    return texts

texts = ingest_and_chunk_pdfs()

Number of documents loaded: 645
Document 1:
Centre Number Candidate NumberWrite your name here
Surname Other names
Total MarksPaper Reference
Turn over     Pearson 
Edexcel GCSE
*P44728A0112* P44728A
©2015 Pearson Education Ltd.
4/1/1/1Instructions
• Use black  ink or ball-point pen.• Fill in the boxes  at the top of this page with your name,  
 centre number and candidate number.• Answer all questions.•  Answer the questions in the spaces provided  
– there may be more space than you need .
Information
• The total mark for this paper is 
Document 2:
2*P44728A0212*Answer ALL questions.
1 Study Section 1 (pages 2, 3 and 4) of the Resource Booklet and answer the following 
questions. 
 (a) (i) Define the term aquifer .
(2)
............................................................................................................................... ............................................................................................................................... .............

In [16]:
embedding_store_path = os.path.join(root_dir, "embedding_store")

def store_embeddings(docs, embeddings, store_name, path):
    vector_store = FAISS.from_documents(docs, embeddings)
    with open(os.path.join(path, f"faiss_{store_name}.pkl"), "wb") as f:
        pickle.dump(vector_store, f)

def load_embeddings(store_name, path):
    with open(os.path.join(path, f"faiss_{store_name}.pkl"), "rb") as f:
        vector_store = pickle.load(f)
    return vector_store

def initialize_huggingface_embeddings(model_name="hkunlp/instructor-xl", device="cuda"):
    return HuggingFaceInstructEmbeddings(model_name=model_name, model_kwargs={"device": device})

instructor_embeddings = initialize_huggingface_embeddings()


/usr/local/lib/python3.10/dist-packages/InstructorEmbedding/instructor.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


.gitattributes:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

2_Dense/config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

load INSTRUCTOR_Transformer
max_seq_length  512


In [ ]:

def retrieve_relevant_chunks(question, vector_store, num_chunks=5):
    # Retrieve similar chunks from the vector store
    docs = vector_store.similarity_search(question, k=num_chunks)
    return docs

def format_prompt(question, chunks):
    context = "\n".join([chunk.page_content for chunk in chunks])
    prompt = f"Provide an answer to the following question using only the context provided: {question}? " \
             f"If you cannot answer this question from the information provided, respond with 'There is insufficient information to answer this question.'\n\n{context}"
    return prompt

def gen_answer(prompt, model_name='llm', max_length=200, temperature=0.7):
    generator = pipeline('text-generation', model=model_name)
    response = generator(prompt, max_length=max_length, temperature=temperature, num_return_sequences=1)
    return response[0]['generated_text'].strip()




In [ ]:
def main(question):
    # Load your vector store (ensure it's already populated with embeddings)
    vector_store = load_embeddings("embedding_store", root_dir)

    # Retrieve relevant chunks based on the question
    relevant_chunks = retrieve_relevant_chunks(question, vector_store)

    # Format the prompt for the LLM
    prompt = format_prompt(question, relevant_chunks)

    # Generate the answer using the LLM
    answer = gen_answer(prompt)
    return answer

# Example usage
question = "What are the main causes of climate change?"
answer = main(question)
print(answer)